# Review Streaming (Kafka) - Functionized

Each function is defined in its own cell for easy conversion into a PySpark job later.


## Imports and shared Spark types


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


## get_env: read environment variables with defaults


In [2]:
def get_env(name: str, default: str) -> str:
    """Return the environment variable value or a default."""
    return os.getenv(name, default)


## build_spark: create a SparkSession


In [3]:
def build_spark(app_name: str = 'review-streaming', mongo_uri: str | None = None) -> SparkSession:
    """Create and return a SparkSession with optional MongoDB configs."""
    builder = SparkSession.builder.appName(app_name)
    if mongo_uri:
        builder = (
            builder
            .config('spark.mongodb.read.connection.uri', mongo_uri)
            .config('spark.mongodb.write.connection.uri', mongo_uri)
        )
    return builder.getOrCreate()


## review_schema: schema for review JSON payloads


In [4]:
def review_schema() -> StructType:
    """Return a StructType describing the review JSON payload."""
    return StructType([
        StructField('review_id', StringType(), True),
        StructField('user_id', StringType(), True),
        StructField('business_id', StringType(), True),
        StructField('stars', IntegerType(), True),
        StructField('useful', IntegerType(), True),
        StructField('funny', IntegerType(), True),
        StructField('cool', IntegerType(), True),
        StructField('text', StringType(), True),
        StructField('date', StringType(), True),
    ])


## build_reviews_stream: read Kafka and parse JSON


In [5]:
def build_reviews_stream(spark: SparkSession):
    """Read Kafka and parse JSON payloads into a streaming DataFrame."""
    kafka_bootstrap = get_env('KAFKA_BOOTSTRAP_SERVERS', 'broker:29092')
    kafka_topic = get_env('KAFKA_TOPIC_REVIEW', 'raw_data_review')

    kafka_df = (
        spark.readStream.format('kafka')
        .option('kafka.bootstrap.servers', kafka_bootstrap)
        .option('subscribe', kafka_topic)
        .option('startingOffsets', 'latest')
        .load()
    )

    schema = review_schema()
    return (
        kafka_df.selectExpr('CAST(value AS STRING) AS json_str')
        .select(from_json(col('json_str'), schema).alias('review'))
        .select('review.*')
    )


## Mongo helpers: read users and businesses


In [6]:
def build_mongo_uri() -> str:
    """Build a MongoDB connection URI from env vars."""
    host = get_env('MONGO_HOST', 'mongodb')
    port = get_env('MONGO_PORT', '27017')
    user = get_env('MONGO_USER', get_env('MONGO_INITDB_ROOT_USERNAME', 'root'))
    password = get_env('MONGO_PASSWORD', get_env('MONGO_INITDB_ROOT_PASSWORD', 'password'))
    auth_db = get_env('MONGO_AUTH_DB', 'admin')
    return f'mongodb://{user}:{password}@{host}:{port}/{auth_db}?authSource={auth_db}'

def load_mongo_collection(spark: SparkSession, collection: str, database: str = None):
    """Load a MongoDB collection into a DataFrame."""
    db = database or get_env('MONGO_DB', 'yelp')
    uri = build_mongo_uri()
    return (
        spark.read.format('mongodb')
        .option('database', db)
        .option('collection', collection)
        .option('uri', uri)
        .load()
    )

def load_users(spark: SparkSession):
    """Load users from MongoDB."""
    return load_mongo_collection(spark, 'users')

def load_businesses(spark: SparkSession):
    """Load businesses from MongoDB."""
    return load_mongo_collection(spark, 'businesses')


## start_memory_sink: write the stream to memory


In [7]:
def start_memory_sink(reviews_stream, query_name: str = 'reviews_stream'):
    """Write the stream to an in-memory table for ad-hoc inspection."""
    return (
        reviews_stream.writeStream
        .format('memory')
        .queryName(query_name)
        .outputMode('append')
        .start()
    )


## show_streamed: view rows from the memory table


In [8]:
def show_streamed(spark: SparkSession, limit: int = 20):
    """Display the most recent rows captured by the memory sink."""
    df = spark.table('reviews_stream')
    df.orderBy(df['date'].desc()).show(limit, truncate=False)


## show_streamed_count: view counts of rows from the memory table

In [9]:
def show_streamed_count(spark: SparkSession):
    """Display the number of rows captured in the memory sink."""
    count = spark.table("reviews_stream").count()
    print(f"reviews_stream count: {count}")


## Sample fields (users + businesses)


## stop_all_streams: stop active streaming queries


In [10]:
def stop_all_streams(spark: SparkSession):
    """Stop every active streaming query attached to this Spark session."""
    for stream in spark.streams.active:
        stream.stop()


## load_users/load_businesses: manual helpers for inspection


In [11]:
def load_users_df(spark: SparkSession):
    """Load users DataFrame for manual inspection."""
    return load_users(spark)

def load_businesses_df(spark: SparkSession):
    """Load businesses DataFrame for manual inspection."""
    return load_businesses(spark)

def show_user_samples(users_df, limit: int = 5):
    """Display a small user sample for sanity checks."""
    users_df.select('user_id', 'name', 'review_count', 'average_stars').show(limit, truncate=False)

def show_business_samples(businesses_df, limit: int = 5):
    """Display a small business sample for sanity checks."""
    businesses_df.select('business_id', 'name', 'city', 'stars').show(limit, truncate=False)


## Manual run (notebook)
Run these cells in order.


In [12]:
mongo_uri = build_mongo_uri()
spark = build_spark(mongo_uri=mongo_uri)
reviews_stream = build_reviews_stream(spark)
query = start_memory_sink(reviews_stream)
query


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/05 08:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/05 08:04:50 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0c0f3369-979b-466e-b405-571af28e7e52. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/05 08:04:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [17]:
# Re-run to refresh
# show_streamed(spark, limit=20)
show_streamed_count(spark)

reviews_stream count: 34


In [15]:
# Load Mongo data for inspection
users_df = load_users_df(spark)
businesses_df = load_businesses_df(spark)
show_user_samples(users_df)
show_business_samples(businesses_df)


+----------------------+----------+------------+-------------+
|user_id               |name      |review_count|average_stars|
+----------------------+----------+------------+-------------+
|T8se5FcRYdebV0K8Mmox-w|Mjcharlene|90          |3.63         |
|xDkoXDdYzwW4IXXw_tDQig|Nicholas  |6           |4.17         |
|vQk6w8G71iKVcEk1I9X4Vg|Fulano    |77          |3.75         |
|G6VzUdl53m8_n0IQCV0pNQ|Tia       |17          |4.29         |
|xDr2ID7Bb0ecmqlAZm4ahA|Martina   |5           |4.2          |
+----------------------+----------+------------+-------------+
only showing top 5 rows
+----------------------+------------------------+-------------+-----+
|business_id           |name                    |city         |stars|
+----------------------+------------------------+-------------+-----+
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Santa Barbara|5.0  |
|mpf3x-BjTdTEA3yCZrAYPw|The UPS Store           |Affton       |3.0  |
|n_0UpQx1hsNbnPUSlodU8w|Famous Footwear         |Brentwood 

In [ ]:
# Stop all running streams
stop_all_streams(spark)
